# Module 02 Homework - Pandas
Using a dataset from _Wine Spectator_, a wine magazine, we will practice the material that we covered in notebook M02_Colab04 to M02_Colab06. This notebook will cover data transformation, grouping, and sorting using pandas.

Created by NAME (SJSU ID)  
Last updated: DATE

In [1]:
csvurl="https://gist.githubusercontent.com/clairehq/" + \
        "79acab35be50eaf1c383948ed3fd1129/raw/407a02139ae1e134992b90b4b2b8c329b3d73a6a/winemag-data-130k-v2.csv"
import pandas as pd
wine = pd.read_csv(csvurl)

In [2]:
wine.head()
wine.columns
wine.shape

(65499, 14)

**Data cleaning**  
Notice that the first column is redundant. Part of data analysis is cleaning and removing redundancy. How would you drop the redundant column inplace, that is overwrite the dataframe.

In [3]:
wine.columns

Index(['Unnamed: 0', 'country', 'description', 'designation', 'points',
       'price', 'province', 'region_1', 'region_2', 'taster_name',
       'taster_twitter_handle', 'title', 'variety', 'winery'],
      dtype='object')

In [4]:
wine = wine.drop(columns=["Unnamed: 0"])

#### Question 1: ####  
What is the mean of the points column?

In [5]:
wine["points"].mean()

np.float64(88.43403716087269)

#### Question 2: ####  
How many countries are present in this dataset? (Only count each country once)

In [6]:
wine["country"].nunique()

41

#### Question 3: ####
How many times does each country appeared in this dataset? Show each country and the corresponding count (show counts in ascending order)

In [7]:
wine["country"].value_counts().sort_values(ascending=True)

,count
country,
Bosnia and Herzegovina,1
Slovakia,1
Armenia,1
Luxembourg,4
Switzerland,4
India,4
Ukraine,5
Macedonia,6
Czech Republic,6


#### Question 4: ####
Create a variable `adjusted_price` containing the adjusted price which is the price subtracted by the average price. *This is called **"centering" transformation** - a method commonly used in the preprocessing step before applying various machine learning algorithms.*

In [8]:
adjusted_price = wine["price"] - wine["price"].mean()

#### Question 5: ####
What is the title of the wine that has the highest points-to-price ratio in the dataset?

In [9]:
# #1 Make the ratio
ratio = wine["points"] / wine["price"]

#2 Find the best ration
best_idx = ratio.idxmax()

#3 Use row index to give title
wine.loc[best_idx, "title"]



'Bandit NV Merlot (California)'

#### Question 6: ####
Create a series `flavor_counts` that contains two values: the number of wines that has the word "tart" in the `description` column and the number of wines that has the word "berries" in the `description` column. The index of the Series should be "Tart" and "Berries" for the corresponding values.

In [10]:
tart_count = wine["description"].str.contains(r"\btart\b", case=False, na=False).sum()
berries_count = wine["description"].str.contains(r"\bberries\b", case=False, na=False).sum()

flavor_counts = pd.Series([tart_count, berries_count], index=["Tart", "Berries"])
flavor_counts

,0
Tart,3086
Berries,1192


#### Question 7: ####
Let's convert the points into simple star ratings. A score of 90 or higher counts as 3 stars, a score of at least 80 but less than 90 is 2 stars. Any other score is 1 star.

Also, any wines from France should automatically get 3 stars, regardless of points.

Add this new column `star_ratings` to the dataframe with the number of stars for each wine in the dataset.

In [11]:
import numpy as np

wine["star_ratings"] = np.select(
    [
        wine["country"].eq("France"),           # rule 1: France always 3 stars
        wine["points"] >= 90,                  # rule 2: 90+ points -> 3 stars
        (wine["points"] >= 80) & (wine["points"] < 90)  # rule 3: 80–89 -> 2 stars
    ],
    [
        3,  # for France
        3,  # for 90+
        2   # for 80–89
    ],
    default=1  # anything else -> 1 star
)

#### Question 8: ####
Who are the most common wine reviewers in the dataset? Create a Series whose index is the taster_twitter_handle category from the dataset, and whose values count how many reviews each person wrote.

In [12]:
reviewer_counts = wine["taster_twitter_handle"].dropna().value_counts()
reviewer_counts

,count
taster_twitter_handle,
@vossroger,13045
@wineschach,7752
@kerinokeefe,5313
@paulgwine,4851
@vboone,4696
@mattkettmann,3035
@JoeCz,2605
@wawinereport,2358
@gordone_cellars,2032


#### Question 9: ####
What combination of countries and varieties are most common? Create a Series whose index is a MultiIndexof {country, variety} pairs. For example, a pinot noir produced in the US should map to {"US", "Pinot Noir"}. Sort the values in the Series in descending order based on wine count.

In [13]:
country_variety_counts = (
    wine.groupby(["country", "variety"])
        .size()
        .sort_values(ascending=False)
)

country_variety_counts

country    variety                 
US         Pinot Noir                  4918
           Cabernet Sauvignon          3649
           Chardonnay                  3412
France     Bordeaux-style Red Blend    2380
Italy      Red Blend                   1870
                                       ... 
Romania    Rosé                           1
US         Ugni Blanc                     1
           Touriga                        1
           Torrontés                      1
Argentina  Merlot-Cabernet Franc          1
Length: 1304, dtype: int64

#### Question 10 #####
Create a Series whose index is reviewers and whose values is the average score given out by that reviewer.  
*Hint:* You will need the `taster_name` and `points` columns.

In [14]:
reviewer_avg_score = wine.groupby("taster_name")["points"].mean()
reviewer_avg_score

,points
taster_name,
Alexander Peartree,86.014286
Anna Lee C. Iijima,88.380506
Anne Krebiehl MW,90.587903
Carrie Dykes,86.644444
Christina Pickard,89.500000
Fiona Adams,87.090909
Jeff Jenssen,88.273504
Jim Gordon,88.604331
Joe Czerwinski,88.519770
